# Throughput of the core draw

**Question.** Does a numpy draw of one dataset per call reach the bar of ADR 0001,
about 500 datasets per second per core at the mean v1 size (N log-uniform on 20 to 15,000,
mean about 2,260 rows, about 20 columns)?
Under the bar, the fallback is a padded torch batch behind the same interface ([#3](https://github.com/adkipnis/tabdgp/issues/3)).

**Setup.** `core.py` beside this notebook holds a throwaway core draw, `drawCore(seed, index, n_obs)`
keyed by `default_rng([seed, index])`: a Cauchy DAG over 20 observed and 0 to 3 latent nodes,
MLP propagation with relative noise and z-scoring at the spec's provisional defaults,
each observed node then rank-matched to a lognormal or thresholded to binary, half and half.
`mlp_share=1` (every node an MLP) is the agreed, conservative case;
`mlp_share=0.5` is the spec's default mechanism mix on average;
`dtype=np.float32` runs that mix in single precision, a lever the spec does not take (its columns are float64).
BLAS is held to one thread, so "one core" means one.
Machine: MacBook Air M3 (4 performance and 4 efficiency cores), 24 GB.

In [1]:
import os

os.environ['VECLIB_MAXIMUM_THREADS'] = '1'  # numpy's macOS wheels use Accelerate
os.environ['OPENBLAS_NUM_THREADS'] = '1'

import cProfile
import platform
import pstats
import sys
import time
import timeit
from concurrent.futures import ProcessPoolExecutor
from itertools import repeat

import numpy as np
from core import drawCore

print(
    platform.platform(),
    platform.machine(),
    'Python',
    sys.version.split()[0],
    'numpy',
    np.__version__,
)

macOS-26.7-arm64-arm-64bit-Mach-O arm64 Python 3.14.7 numpy 2.5.4


## One dataset

Shapes, dtypes, finiteness and determinism under the key.

In [2]:
dataset = drawCore(seed=0, index=1, n_obs=2260)
again = drawCore(seed=0, index=1, n_obs=2260)
dtypes = {str(column.dtype) for column in dataset['columns'].values()}
same = all(
    np.array_equal(dataset['columns'][name], again['columns'][name]) for name in dataset['columns']
)
finite = all(np.isfinite(column).all() for column in dataset['columns'].values())
print('columns', len(dataset['columns']), 'dtypes', dtypes, 'latent', dataset['latent'].shape)
print('edges', int(dataset['parents'].sum()), 'finite', finite, 'bitwise identical', same)

columns 20 dtypes {'float64', 'int64'} latent (2, 2260)
edges 80 finite True bitwise identical True


## One core

Datasets per second over consecutive indices, the median of three repeats.
The drawn row is the verdict's: N drawn once per dataset from LogUniformInt(20, 15000),
the same sizes for every run.

In [3]:
SEED = 0
sizes_rng = np.random.default_rng(2026)
DRAWN = np.exp(sizes_rng.uniform(np.log(20), np.log(15001), 1000)).astype(int)  # (1000,)
print('drawn N: mean', round(DRAWN.mean()), 'median', int(np.median(DRAWN)))


def rate(sizes, mlp_share, dtype=np.float64):
    durations = []
    for _ in range(3):
        start = time.perf_counter()
        for index, n_obs in enumerate(sizes):
            drawCore(SEED, index, int(n_obs), mlp_share, dtype)
        durations.append(time.perf_counter() - start)
    return len(sizes) / np.median(durations)


cases = {
    'N = 100': [100] * 2000,
    'N = 1,000': [1000] * 1000,
    'N = 10,000': [10000] * 100,
    'N = 2,260': [2260] * 500,
    'N drawn': DRAWN,
}
print(
    f'{"case":<12} {"all MLP /s":>12} {"default mix /s":>16} {"mix, float32 /s":>17} {"ms (all MLP)":>14}'
)
for case, sizes in cases.items():
    rates = (rate(sizes, 1.0), rate(sizes, 0.5), rate(sizes, 0.5, np.float32))
    print(
        f'{case:<12} {rates[0]:>12.0f} {rates[1]:>16.0f} {rates[2]:>17.0f} {1000 / rates[0]:>14.2f}'
    )

drawn N: mean 2432 median 676
case           all MLP /s   default mix /s   mix, float32 /s   ms (all MLP)


N = 100               908             1136              1133           1.10


N = 1,000             270              377               486           3.70


N = 10,000             34               49                71          29.49


N = 2,260             136              195               267           7.37


N drawn               125              179               248           7.99


## Where the time goes

A profile at N = 2,260 under the default mix, then the elementwise costs on one MLP layer
of mean width (2,260 × 21) in float64 and float32.
Operators and ufuncs called inside a function count toward that function's own time.

In [4]:
profiler = cProfile.Profile()
profiler.enable()
for index in range(300):
    drawCore(SEED, index, 2260, 0.5)
profiler.disable()
pstats.Stats(profiler).strip_dirs().sort_stats('tottime').print_stats(8)

         682424 function calls (682422 primitive calls) in 1.630 seconds

   Ordered by: internal time
   List reduced from 179 to 8 due to restriction <8>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     4912    0.664    0.000    0.684    0.000 core.py:121(_mlp)
     2940    0.263    0.000    0.263    0.000 {method 'argsort' of 'numpy.ndarray' objects}
     6000    0.117    0.000    0.117    0.000 {method 'sort' of 'numpy.ndarray' objects}
      300    0.104    0.000    0.962    0.003 core.py:98(_propagate)
     2940    0.094    0.000    0.480    0.000 core.py:151(_rankMatch)
     3060    0.057    0.000    0.057    0.000 {method 'partition' of 'numpy.ndarray' objects}
    11378    0.051    0.000    0.083    0.000 _methods.py:148(_var)
    64772    0.041    0.000    0.041    0.000 {method 'reduce' of 'numpy.ufunc' objects}




In [5]:
rng = np.random.default_rng(0)
layer = rng.standard_normal((2260, 21))  # (N, hidden)
weights = rng.standard_normal((21, 21))  # (hidden, hidden)
operations = {
    'matmul': 'x @ w',
    'tanh': 'np.tanh(x)',
    'sin': 'np.sin(x)',
    'relu': 'np.maximum(x, 0)',
    'argsort of a column': 'np.argsort(x[:, 0])',
}
print(f'{"operation":<22} {"float64 us":>11} {"float32 us":>11}')
for name, statement in operations.items():
    timings = []
    for dtype in (np.float64, np.float32):
        names = {'np': np, 'x': layer.astype(dtype), 'w': weights.astype(dtype)}
        timings.append(
            min(timeit.repeat(statement, globals=names, number=2000, repeat=3)) / 2000 * 1e6
        )
    print(f'{name:<22} {timings[0]:>11.1f} {timings[1]:>11.1f}')

operation               float64 us  float32 us


matmul                        29.8        19.9


tanh                         125.1        37.8


sin                          200.8        57.7
relu                          13.9        13.8


argsort of a column           23.9        23.5


## A process pool over indices

`ProcessPoolExecutor` with spawned workers, the pool started and warmed before the clock,
the datasets returned to the parent (as a dataloader's workers would), the drawn sizes.

In [6]:
def poolRate(workers, mlp_share):
    with ProcessPoolExecutor(max_workers=workers) as pool:
        list(pool.map(drawCore, repeat(SEED), range(workers), repeat(100), repeat(mlp_share)))
        start = time.perf_counter()
        sizes = [int(n_obs) for n_obs in np.tile(DRAWN, 2)]  # 2000 datasets
        list(
            pool.map(
                drawCore, repeat(SEED), range(len(sizes)), sizes, repeat(mlp_share), chunksize=16
            )
        )
        return len(sizes) / (time.perf_counter() - start)


print(f'{"workers":>7} {"all MLP /s":>12} {"default mix /s":>16} {"speed-up (all MLP)":>20}')
pool_rates = {}
for workers in (1, 2, 4, 8):
    pool_rates[workers] = (poolRate(workers, 1.0), poolRate(workers, 0.5))
    speedup = pool_rates[workers][0] / pool_rates[1][0]
    print(
        f'{workers:>7} {pool_rates[workers][0]:>12.0f} {pool_rates[workers][1]:>16.0f} {speedup:>20.1f}'
    )

workers   all MLP /s   default mix /s   speed-up (all MLP)


      1          124              172                  1.0


      2          235              328                  1.9


      4          416              579                  3.4


      8          533              751                  4.3


## Verdict

**Under the bar.** At the mean v1 size, one core draws about 130 datasets per second with every node an MLP,
about 180 under the spec's default mechanism mix and about 250 under that mix in float32:
two to four times short of 500.

- **The draw is arithmetic-bound, not interpreter-bound.**
  At N = 100 a draw costs about 1 ms, the per-dataset overhead of the Python loop over nodes.
  From 1,000 rows up, the cost grows with N (about 30 ms at 10,000).
  At N = 2,260 the MLPs (matrix products and the tanh and sin activations) take about 40 % of the time,
  rank matching (argsort and sort) about 30 % and the loop itself about 6 %.
- **float32 buys about 1.4 times.** The activations run three to four times faster in single precision,
  but sorting and the reductions do not, and the spec's columns are float64.
- **A pool of 8 workers gives 4.3 times**, about 540 to 750 datasets per second on the M3's
  4 performance and 4 efficiency cores.
- **The fallback as worded does not address this miss.** A padded torch batch on CPU does the same arithmetic,
  plus the padding of N from 20 to 15,000 up to the batch's largest dataset, so it should not buy two to four times per core (inferred, torch not timed here);
  a GPU batch could, the road ADR 0001 rejected for determinism and launch overhead.

The decision taken on these numbers is recorded in `lab/README.md`.